<a href="https://colab.research.google.com/github/hanamonem/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hanamonem/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes
### Rule

I will prioritize content that has not been updated recently and is showing a declining trend.

Signals:
1. Days since last update
2. Trend percentage

Reason code:
STALE_DOWNTREND

Action:
REFRESH_CONTENT


In [17]:
!git clone https://github.com/hanamonem/flyrank-ml-internship.git

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 137, done.
remote: Counting objects: 100% (137/137), done.
remote: Compressing objects: 100% (93/93), done.
remote: Total 137 (delta 46), reused 95 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (137/137), 1.88 MiB | 7.51 MiB/s, done.
Resolving deltas: 100% (46/46), done.


In [18]:
%cd flyrank-ml-internship

/content/flyrank-ml-internship


In [19]:
!find . -name "*.csv"

./outputs/refresh_queue_sample.csv
./data/raw/content_refresh_anonymized.csv


In [20]:
import pandas as pd

df = pd.read_csv("./data/raw/content_refresh_anonymized.csv")

print(df.shape)


(30000, 44)


In [21]:
df.head()


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [22]:
df[['content_age_days','days_since_last_update','trend_pct']].describe()

,content_age_days,days_since_last_update,trend_pct
count,30000.00000,30000.000000,26612.000000
mean,256.16780,46.098300,-4.785969
std,132.70793,42.078709,473.861780
min,90.00000,1.000000,-100.000000
25%,132.00000,20.000000,-62.600000
50%,236.00000,20.000000,-33.500000
75%,333.00000,104.000000,0.000000
max,564.00000,373.000000,44900.000000


In [23]:
df.columns.tolist()

['content_id',
 'client_id',
 'search_volume',
 'competition',
 'competition_level',
 'cpc',
 'content_type',
 'main_intent',
 'word_count',
 'char_count',
 'provider_used',
 'model_used',
 'impressions_90d',
 'clicks_90d',
 'pageviews_90d',
 'sessions_90d',
 'users_90d',
 'engaged_sessions_90d',
 'ai_sessions_90d',
 'scroll_events_90d',
 'days_with_impressions',
 'days_with_sessions',
 'impressions_last_30d',
 'clicks_last_30d',
 'sessions_last_30d',
 'impressions_prev_30d',
 'clicks_prev_30d',
 'sessions_prev_30d',
 'content_age_days',
 'age_tier',
 'age_tier_order',
 'days_since_last_update',
 'freshness_tier',
 'word_count_tier',
 'char_count_tier',
 'ctr',
 'avg_position',
 'engagement_rate',
 'scroll_rate',
 'ai_traffic_pct',
 'impression_tier',
 'position_tier',
 'trend_direction',
 'trend_pct']

### Signal Check 1: Days Since Last Update

I grouped content by days since last update and compared the average trend percentage.

Pages updated within the last 30 days had a slightly positive average trend (+0.78%), while pages that had not been updated for 91–180 days showed a much more negative average trend (-15.68%).

n = 30000

Verdict: CONFIRMED

The observed pattern suggests that content becomes more likely to decline when it has not been updated for longer periods.

In [24]:
# Signal Check 1

import pandas as pd

df["update_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[0, 30, 90, 180, 400],
    labels=["0-30", "31-90", "91-180", "180+"]
)

signal_1 = (
    df.groupby("update_bucket")
      .agg(
          avg_trend=("trend_pct", "mean"),
          n=("content_id", "count")
      )
)

print(signal_1)

               avg_trend      n
update_bucket                  
0-30            0.784405  20480
31-90          -7.373054    175
91-180        -15.683224   9171
180+           -6.781203    174


/tmp/ipykernel_1174/957390511.py:12: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("update_bucket")


### Signal Check 2: Trend Direction

I grouped content by trend direction and compared average clicks.

Result:
- Stable pages had the highest average clicks (34.14).
- Down-trending pages averaged 13.36 clicks.
- Up-trending pages averaged 13.94 clicks.

n = 30000

Verdict: CONFIRMED

Observed performance differs by trend direction, suggesting that trend is a meaningful signal for prioritizing refresh opportunities.

In [25]:
signal_2 = (
    df.groupby("trend_direction")
      .agg(
          avg_clicks=("clicks_90d", "mean"),
          n=("content_id", "count")
      )
      .reset_index()
)

print(signal_2)

  trend_direction  avg_clicks      n
0            down   13.359242  16262
1            flat    0.055556   1152
2             new    0.385957   2236
3          stable   34.142402   5962
4              up   13.944394   4388


## 2. Build the ranked queue (writes the CSV)
## Ranked Queue

I created a baseline action score for content refresh opportunities.

The score combines:

- Days since last update (staleness)
- Negative trend percentage

Higher scores indicate content that is more likely to benefit from a refresh.

Reason code:
STALE_DOWNTREND

Action:
REFRESH_CONTENT

The ranked queue was written to:

work/outputs/baseline_action_score.csv

In [26]:
import numpy as np
import pandas as pd
from pathlib import Path

# copy dataframe
queue = df.copy()

# normalize staleness (older = higher score)
queue["staleness_score"] = (
    queue["days_since_last_update"]
    / queue["days_since_last_update"].max()
)

# normalize decline
queue["decline_score"] = (
    (-queue["trend_pct"]).clip(lower=0)
)

queue["decline_score"] = (
    queue["decline_score"]
    / queue["decline_score"].max()
)

# final score
queue["score"] = (
    0.6 * queue["staleness_score"]
    + 0.4 * queue["decline_score"]
)

# reason code and action
queue["reason_code"] = "STALE_DOWNTREND"
queue["action"] = "REFRESH_CONTENT"

# rank
queue = queue.sort_values(
    "score",
    ascending=False
).reset_index(drop=True)

# keep output columns
output = queue[
    [
        "content_id",
        "score",
        "reason_code",
        "action"
    ]
]

# write csv
Path("work/outputs").mkdir(
    parents=True,
    exist_ok=True
)

output.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print(output.head(10))
print("\nRows:", len(output))

             content_id     score      reason_code           action
0  content_f6fdf87348f6  1.000000  STALE_DOWNTREND  REFRESH_CONTENT
1  content_1b4ec72dafd4  0.998391  STALE_DOWNTREND  REFRESH_CONTENT
2  content_55a5b1c46474  0.954000  STALE_DOWNTREND  REFRESH_CONTENT
3  content_7a888d3d99c8  0.903485  STALE_DOWNTREND  REFRESH_CONTENT
4  content_94991fe6268c  0.903485  STALE_DOWNTREND  REFRESH_CONTENT
5  content_ab18b5811c02  0.890617  STALE_DOWNTREND  REFRESH_CONTENT
6  content_84d12054c0c0  0.889008  STALE_DOWNTREND  REFRESH_CONTENT
7  content_02b0d6e30129  0.885885  STALE_DOWNTREND  REFRESH_CONTENT
8  content_ccfb4d0227b1  0.884182  STALE_DOWNTREND  REFRESH_CONTENT
9  content_df1fa766cac2  0.879808  STALE_DOWNTREND  REFRESH_CONTENT

Rows: 30000


## 3. Top-20 review
## Top-20 Review

### Rank 1
Action: REFRESH_CONTENT

Reason Code: STALE_DOWNTREND

Confidence: High. The page has not been updated for a long time and shows a strong negative trend.

What would make it wrong:
The decline may be caused by seasonality rather than outdated content.

### Rank 2
Action: REFRESH_CONTENT

Reason Code: STALE_DOWNTREND

Confidence: High. Both staleness and declining trend support refresh.

What would make it wrong:
Traffic changes could be driven by search demand shifts.

### Rank 3
Action: REFRESH_CONTENT

Reason Code: STALE_DOWNTREND

Confidence: Moderate. The page is old and trending down.

What would make it wrong:
The page may still satisfy user intent despite declining traffic.

...

In [27]:
top20 = queue.head(20)

top20[
    [
        "content_id",
        "score",
        "reason_code",
        "action",
        "days_since_last_update",
        "trend_pct"
    ]
]

,content_id,score,reason_code,action,days_since_last_update,trend_pct
0,content_f6fdf87348f6,1.000000,STALE_DOWNTREND,REFRESH_CONTENT,373,-100.0
1,content_1b4ec72dafd4,0.998391,STALE_DOWNTREND,REFRESH_CONTENT,372,-100.0
2,content_55a5b1c46474,0.954000,STALE_DOWNTREND,REFRESH_CONTENT,373,-88.5
3,content_7a888d3d99c8,0.903485,STALE_DOWNTREND,REFRESH_CONTENT,313,-100.0
4,content_94991fe6268c,0.903485,STALE_DOWNTREND,REFRESH_CONTENT,313,-100.0
5,content_ab18b5811c02,0.890617,STALE_DOWNTREND,REFRESH_CONTENT,305,-100.0
6,content_84d12054c0c0,0.889008,STALE_DOWNTREND,REFRESH_CONTENT,304,-100.0
7,content_02b0d6e30129,0.885885,STALE_DOWNTREND,REFRESH_CONTENT,313,-95.6
8,content_ccfb4d0227b1,0.884182,STALE_DOWNTREND,REFRESH_CONTENT,301,-100.0
9,content_df1fa766cac2,0.879808,STALE_DOWNTREND,REFRESH_CONTENT,304,-97.7


## 4. Weak picks + leakage check

## Weak Picks

Some lower-ranked refresh recommendations appear weaker than the top-ranked items.

Pages with limited impressions or sessions may receive a high score because of content age or a negative trend, even though the observed decline may be caused by small sample sizes.

Pages with strong historical performance but a short-term decline may also be incorrectly prioritized for refresh. Additional qualitative review would be helpful before taking action.

## Leakage Check

I did not use any product flags, labels, or reason codes as input features.

I only used current content attributes and performance signals available in the dataset, including days_since_last_update and trend_pct.

No future-window information was used when calculating the baseline score.

This baseline should be considered a directional, decision-support ranking rather than a prediction of future outcomes.

In [28]:
# Review lowest-confidence candidates

weak_picks = queue.tail(10)[
    [
        "content_id",
        "score",
        "days_since_last_update",
        "trend_pct"
    ]
]

print("Potential weak picks:")
display(weak_picks)

print("\nLeakage check")

used_features = [
    "days_since_last_update",
    "trend_pct"
]

print("Features used:", used_features)

forbidden_features = [
    "reason_code",
    "action"
]

print("No product flags, labels, or reason codes used as model inputs.")
print("No future-window variables were used.")

Potential weak picks:


,content_id,score,days_since_last_update,trend_pct
29990,content_0934cd438dd6,NaN,8,NaN
29991,content_f452475bff46,NaN,20,NaN
29992,content_cd850fb019b1,NaN,20,NaN
29993,content_b1d45033b059,NaN,20,NaN
29994,content_a3af3b8346d8,NaN,6,NaN
29995,content_179533212cd0,NaN,20,NaN
29996,content_92a5d2709aa9,NaN,8,NaN
29997,content_2dfd17269502,NaN,20,NaN
29998,content_23dce6a656e6,NaN,20,NaN
29999,content_c322796023c8,NaN,20,NaN



Leakage check
Features used: ['days_since_last_update', 'trend_pct']
No product flags, labels, or reason codes used as model inputs.
No future-window variables were used.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.